In [3]:
import pandas as pd
from pathlib import Path

# 1. Таргет (оставляем как было)
target_path = 'archive/alfabattle2_train_target.csv' 
target = pd.read_csv(target_path)
print("--- ТАРГЕТ ---")
print(f"Размер: {target.shape}")
print(f"Доля дефолтов (flag=1): {target['flag'].mean():.4f}\n")

# 2. Транзакции (указываем полный путь с вложенной папкой)
trx_dir = Path('archive/alfabattle2_train_transactions_contest/train_transactions_contest')
trx_files = list(trx_dir.glob('*.parquet'))

if len(trx_files) > 0:
    print(f"--- ТРАНЗАКЦИИ ---")
    print(f"Найдено файлов: {len(trx_files)}")
    
    # Читаем самый первый файл
    sample_trx = pd.read_parquet(trx_files[0])
    print(f"Размер одного файла: {sample_trx.shape}")
    print("\nСписок всех колонок (19 признаков):")
    print(sample_trx.columns.tolist())
        
    print("\nПервые 3 строки для наглядности:")
    print(sample_trx.head(3))
else:
    print(f"Файлы не найдены в папке: {trx_dir.absolute()}")

--- ТАРГЕТ ---
Размер: (963811, 3)
Доля дефолтов (flag=1): 0.0276

--- ТРАНЗАКЦИИ ---
Найдено файлов: 50
Размер одного файла: (5408648, 20)

Список всех колонок (19 признаков):
['app_id', 'amnt', 'currency', 'operation_kind', 'card_type', 'operation_type', 'operation_type_group', 'ecommerce_flag', 'payment_system', 'income_flag', 'mcc', 'country', 'city', 'mcc_category', 'day_of_week', 'hour', 'days_before', 'weekofyear', 'hour_diff', 'transaction_number']

Первые 3 строки для наглядности:
   app_id      amnt  currency  operation_kind  card_type  operation_type  \
0       0  0.465425         1               4         98               4   
1       0  0.000000         1               2         98               7   
2       0  0.521152         1               2         98               3   

   operation_type_group  ecommerce_flag  payment_system  income_flag  mcc  \
0                     2               3               7            3    2   
1                     1               3       

In [4]:
# Создадим функцию, которая превращает транзакции в признаки
def create_basic_features(trx_df):
    # Группируем по клиенту
    features = trx_df.groupby('app_id').agg(
        # 1. Статистики по сумме покупки (amnt)
        amnt_mean=('amnt', 'mean'),
        amnt_std=('amnt', 'std'),
        amnt_max=('amnt', 'max'),
        amnt_sum=('amnt', 'sum'),
        
        # 2. Количество транзакций
        trx_count=('amnt', 'count'),
        
        # 3. Количество уникальных MCC-категорий (насколько разнообразны траты)
        unique_mcc=('mcc_category', 'nunique'),
        
        # 4. Средняя и максимальная глубина истории (давность транзакций)
        days_before_mean=('days_before', 'mean'),
        days_before_min=('days_before', 'min'),
        
        # 5. Доля онлайн-покупок (ecommerce_flag)
        ecommerce_share=('ecommerce_flag', 'mean'),
        
        # 6. Среднее количество дней между транзакциями (hour_diff)
        hour_diff_mean=('hour_diff', 'mean'),
        
        # 7. Максимальный порядковый номер транзакции (сколько всего транзакций у клиента)
        max_trx_number=('transaction_number', 'max')
    ).reset_index()
    
    # Добавляем признак "Ночные транзакции" (если час >= 22 или <= 6)
    trx_df['is_night'] = ((trx_df['hour'] >= 22) | (trx_df['hour'] <= 6)).astype(int)
    night_counts = trx_df.groupby('app_id')['is_night'].sum().reset_index(name='night_trx_count')
    
    # Добавляем признак "Выходные" (день недели 5 или 6)
    trx_df['is_weekend'] = trx_df['day_of_week'].isin([5, 6]).astype(int)
    weekend_counts = trx_df.groupby('app_id')['is_weekend'].sum().reset_index(name='weekend_trx_count')
    
    # Объединяем все признаки
    features = features.merge(night_counts, on='app_id', how='left')
    features = features.merge(weekend_counts, on='app_id', how='left')
    
    return features

# Возьмем только первый файл для проверки (чтобы не ждать долго)
print("Читаем 1-й файл...")
sample_trx = pd.read_parquet(trx_files[0])

print("Генерируем признаки...")
sample_features = create_basic_features(sample_trx)

print(f"Получилось признаков: {sample_features.shape[1] - 1} (плюс app_id)")
print(sample_features.head())

Читаем 1-й файл...
Генерируем признаки...
Получилось признаков: 13 (плюс app_id)
   app_id  amnt_mean  amnt_std  amnt_max    amnt_sum  trx_count  unique_mcc  \
0       0   0.386645  0.094032  0.609342   69.982751        181          12   
1       1   0.335351  0.078044  0.578477  119.384783        356          15   
2       2   0.306107  0.087890  0.496822   70.098407        229           9   
3       3   0.330808  0.098987  0.572565   22.164130         67           9   
4       4   0.412301  0.128579  0.646036   48.239197        117           4   

   days_before_mean  days_before_min  ecommerce_share  hour_diff_mean  \
0        156.944751               14         1.099448       44.613260   
1        167.320225                1         1.171348       24.053371   
2        171.196507                2         1.423581       36.655022   
3        166.805970               20         1.208955      109.104478   
4        200.888889                1         1.367521       71.837607   

   ma

In [5]:
import time

# Список для хранения признаков из каждого файла
all_features = []

print(f"Начинаем обработку {len(trx_files)} файлов...")
start_time = time.time()

# Проходимся по всем файлам
for i, file_path in enumerate(trx_files):
    # Читаем файл
    trx_df = pd.read_parquet(file_path)
    
    # Генерируем признаки (используем нашу функцию)
    features_df = create_basic_features(trx_df)
    
    # Добавляем в общий список
    all_features.append(features_df)
    
    # Выводим прогресс, чтобы вы не думали, что код завис
    if (i + 1) % 5 == 0 or (i + 1) == len(trx_files):
        print(f"Обработано файлов: {i + 1}/{len(trx_files)}")
        
    # Очищаем память (удаляем тяжелый файл транзакций)
    del trx_df
    del features_df

# Объединяем все признаки в один датафрейм
print("Объединяем все признаки...")
full_features = pd.concat(all_features, ignore_index=True)

# ВАЖНО: Один клиент мог быть в разных файлах!
# Поэтому нам нужно сгруппировать еще раз по app_id и взять среднее/максимум
print("Финальная группировка по клиентам (устраняем дубликаты)...")
# Для числовых признаков берем среднее (mean), для счетчиков - сумму (sum)
# Но чтобы не усложнять, пока просто возьмем mean
final_features = full_features.groupby('app_id').mean().reset_index()

print(f"Готово! Потрачено времени: {time.time() - start_time:.2f} секунд")
print(f"Итоговый размер таблицы признаков: {final_features.shape}")
print(final_features.head())

Начинаем обработку 50 файлов...
Обработано файлов: 5/50
Обработано файлов: 10/50
Обработано файлов: 15/50
Обработано файлов: 20/50
Обработано файлов: 25/50
Обработано файлов: 30/50
Обработано файлов: 35/50
Обработано файлов: 40/50
Обработано файлов: 45/50
Обработано файлов: 50/50
Объединяем все признаки...
Финальная группировка по клиентам (устраняем дубликаты)...
Готово! Потрачено времени: 54.29 секунд
Итоговый размер таблицы признаков: (963811, 14)
   app_id  amnt_mean  amnt_std  amnt_max    amnt_sum  trx_count  unique_mcc  \
0       0   0.386645  0.094032  0.609342   69.982751      181.0        12.0   
1       1   0.335351  0.078044  0.578477  119.384783      356.0        15.0   
2       2   0.306107  0.087890  0.496822   70.098407      229.0         9.0   
3       3   0.330808  0.098987  0.572565   22.164130       67.0         9.0   
4       4   0.412301  0.128579  0.646036   48.239197      117.0         4.0   

   days_before_mean  days_before_min  ecommerce_share  hour_diff_mean 

In [ ]:
# Объединяем признаки и таргет по app_id
# Используем how='left', чтобы оставить всех клиентов из таргета
dataset = target.merge(final_features, on='app_id', how='left')

# Проверим, что получилось
print(f"Размер итогового датасета: {dataset.shape}")
print(f"Колонки: {dataset.columns.tolist()}")

# Посмотрим, есть ли пропуски (клиенты без транзакций)
print(f"Пропуски в признаках: {dataset.isnull().sum().sum()}")